In [1]:
year = 2007
month = 1

In [2]:
# Parameters
year = 1998
month = 4


In [3]:
import numpy as np
import pandas as pd
import xarray as xr
import os, pathlib, stat, textwrap
import calendar
import datetime
from datetime import date

### URLs

In [4]:
# Ufiles = "https://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Ufiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridU"
Vfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridV"
Wfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridW"
Tfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridT"
Sfiles = "dap2://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1-daily-gridS"
# #mesh url
# Zgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_zgr.nc"
# Hgr = "http://tds.mercator-ocean.fr/thredds/dodsC/glorys12v1/glorys12v1-pgnstatics/PSY4V3R1_mesh_hgr.nc"

### Environment 

In [5]:
os.environ["NETRC"] = "/home/b/b383184/.netrc"

### Mesh

In [6]:
ds_Zgr = xr.open_dataset('../data/Zgr_mesh.nc')
ds_Zgr

<xarray.Dataset> Size: 555MB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/13)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    mbathy        (t, y, x) int16 26MB ...
    hdept         (t, y, x) float64 106MB ...
    ...            ...
    e3t_ps        (t, y, x) float64 106MB ...
    e3w_ps        (t, y, x) float64 106MB ...
    gdept_0       (t, z) float64 400B ...
    gdepw_0       (t, z) float64 400B ...
    e3t_0         (t, z) float64 400B ...
    e3w_0         (t, z) float64 400B ...
Attributes:
    file_name:            mesh_zgr.nc
    TimeStamp:            03/02/2016 10:24:41 -0000
    Unlimited_Dimension:  t

In [7]:
ds_Hgr = xr.open_dataset('../data/Hgr_mesh.nc')
ds_Hgr

<xarray.Dataset> Size: 1GB
Dimensions:       (y: 3059, x: 4322, z: 50, t: 1)
Dimensions without coordinates: y, x, z, t
Data variables: (12/21)
    nav_lon       (y, x) float32 53MB ...
    nav_lat       (y, x) float32 53MB ...
    nav_lev       (z) float32 200B ...
    time_counter  (t) float64 8B ...
    glamt         (t, y, x) float32 53MB ...
    glamu         (t, y, x) float32 53MB ...
    ...            ...
    e1f           (t, y, x) float64 106MB ...
    e2t           (t, y, x) float64 106MB ...
    e2u           (t, y, x) float64 106MB ...
    e2v           (t, y, x) float64 106MB ...
    e2f           (t, y, x) float64 106MB ...
    ff            (t, y, x) float64 106MB ...
Attributes:
    file_name:            mesh_hgr.nc
    TimeStamp:            03/02/2016 10:24:55 -0000
    Unlimited_Dimension:  t

### Functions

In [8]:
lon0, lon1 = -95, 10
lat0, lat1 = -10, 30

# 1) Use T-point lon/lat
lonT = ds_Hgr.glamt.isel(t=0)
latT = ds_Hgr.gphit.isel(t=0)

# 2) Build boolean mask for your box
mask = (lonT >= lon0) & (lonT <= lon1) & (latT >= lat0) & (latT <= lat1)

# 3) Get index ranges
yy, xx = np.where(mask.values)

y0, y1 = int(yy.min()), int(yy.max())
x0, x1 = int(xx.min()), int(xx.max())

x0, x1, y0, y1

(2305, 3565, 1374, 1873)

In [9]:
last_day = calendar.monthrange(year, month)[1]
start_date = datetime.datetime(year, month, 1)
end_date = datetime.datetime(year, month, last_day)

print(end_date.strftime("%Y-%m-%d"))

1998-04-30


In [10]:
def glorys_days(start_date, end_date):
    return pd.date_range(start=start_date, end=end_date, freq="D") + pd.Timedelta(hours=12)

days = glorys_days(start_date.strftime("%Y-%m-%d")
                   , end_date.strftime("%Y-%m-%d"))

In [11]:
def download_MERCATOR(url, varname, starts, ends, x0, x1, y0, y1, output_file):

    from tqdm import tqdm
    import xarray as xr
    
    parts = []
    for tt in tqdm(range(len(days)//2)):
        da = (
            xr.open_dataset(url, engine="pydap", mask_and_scale=False, decode_cf=True)[varname]
            .sortby("time_counter")
            .isel(x=slice(x0, x1), y=slice(y0, y1))
            .sel(time_counter=slice(starts[tt], ends[tt]))
            .astype("float32")
            .load()
        )
        parts.append(da)
    
    da_all = xr.concat(parts, dim="time_counter")
    da_all.to_dataset(name=varname).to_netcdf(output_file, unlimited_dims=["time_counter"])
    print(f"Saved {output_file}")

In [12]:
starts = days[0::2]
ends = days[1::2].tolist()  
ends[-1] = days[-1]
ends

for tt in  range(len(days)//2):
    print('start_date '+str(starts[tt]))
    print('end_date '+str(ends[tt]))

start_date 1998-04-01 12:00:00
end_date 1998-04-02 12:00:00
start_date 1998-04-03 12:00:00
end_date 1998-04-04 12:00:00
start_date 1998-04-05 12:00:00
end_date 1998-04-06 12:00:00
start_date 1998-04-07 12:00:00
end_date 1998-04-08 12:00:00
start_date 1998-04-09 12:00:00
end_date 1998-04-10 12:00:00
start_date 1998-04-11 12:00:00
end_date 1998-04-12 12:00:00
start_date 1998-04-13 12:00:00
end_date 1998-04-14 12:00:00
start_date 1998-04-15 12:00:00
end_date 1998-04-16 12:00:00
start_date 1998-04-17 12:00:00
end_date 1998-04-18 12:00:00
start_date 1998-04-19 12:00:00
end_date 1998-04-20 12:00:00
start_date 1998-04-21 12:00:00
end_date 1998-04-22 12:00:00
start_date 1998-04-23 12:00:00
end_date 1998-04-24 12:00:00
start_date 1998-04-25 12:00:00
end_date 1998-04-26 12:00:00
start_date 1998-04-27 12:00:00
end_date 1998-04-28 12:00:00
start_date 1998-04-29 12:00:00
end_date 1998-04-30 12:00:00


### Data download

In [13]:
U_out = f'U_{start_date.strftime("%Y-%m")}.nc'
V_out = f'V_{start_date.strftime("%Y-%m")}.nc'
W_out = f'W_{start_date.strftime("%Y-%m")}.nc'
T_out = f'T_{start_date.strftime("%Y-%m")}.nc'
S_out = f'S_{start_date.strftime("%Y-%m")}.nc'

outpath = '/work/bk1450/b383184/Amazon/Mercator/data/variables/'

In [14]:
#U 
download_MERCATOR(
    Ufiles, "vozocrtx", starts, ends, x0, x1, y0, y1,outpath+U_out
)

  0%|                                                                                                   | 0/15 [00:00<?, ?it/s]

  7%|██████                                                                                    | 1/15 [03:50<53:52, 230.90s/it]

 13%|████████████                                                                              | 2/15 [04:20<24:24, 112.68s/it]

 20%|██████████████████▏                                                                        | 3/15 [05:23<17:56, 89.73s/it]

 27%|████████████████████████▎                                                                  | 4/15 [05:42<11:22, 62.07s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████            | 13/15 [11:42<00:51, 25.80s/it]

 93%|████████████████████████████████████████████████████████████████████████████████████      | 14/15 [12:28<00:32, 32.03s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [12:48<00:00, 28.55s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [12:48<00:00, 51.26s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/U_1998-04.nc


In [15]:
download_MERCATOR(
    Vfiles, "vomecrty", starts, ends, x0, x1, y0, y1,outpath+V_out 
)

  0%|                                                                                                   | 0/15 [00:00<?, ?it/s]

  7%|██████                                                                                    | 1/15 [02:52<40:10, 172.20s/it]

 13%|████████████▏                                                                              | 2/15 [03:13<18:01, 83.16s/it]

 20%|██████████████████▏                                                                        | 3/15 [03:45<11:59, 59.96s/it]

 27%|████████████████████████▎                                                                  | 4/15 [04:04<08:02, 43.86s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████            | 13/15 [07:36<00:45, 22.84s/it]

 93%|████████████████████████████████████████████████████████████████████████████████████      | 14/15 [07:56<00:21, 21.98s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [08:27<00:00, 24.70s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [08:27<00:00, 33.83s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/V_1998-04.nc


In [16]:
download_MERCATOR(
    Wfiles, "vovecrtz", starts, ends, x0, x1, y0, y1,outpath+W_out 
)

  0%|                                                                                                   | 0/15 [00:00<?, ?it/s]

  7%|██████                                                                                    | 1/15 [02:08<29:56, 128.33s/it]

 13%|████████████▏                                                                              | 2/15 [02:27<13:49, 63.84s/it]

 20%|██████████████████▏                                                                        | 3/15 [02:46<08:43, 43.60s/it]

 27%|████████████████████████▎                                                                  | 4/15 [03:07<06:19, 34.47s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████            | 13/15 [08:01<01:15, 37.63s/it]

 93%|████████████████████████████████████████████████████████████████████████████████████      | 14/15 [08:22<00:32, 32.52s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [08:54<00:00, 32.39s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [08:54<00:00, 35.62s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/W_1998-04.nc


In [17]:
download_MERCATOR(
    Tfiles, "votemper", starts, ends, x0, x1, y0, y1,outpath+T_out
)

  0%|                                                                                                   | 0/15 [00:00<?, ?it/s]

  7%|██████                                                                                     | 1/15 [01:02<14:32, 62.29s/it]

 13%|████████████▏                                                                              | 2/15 [01:23<08:15, 38.15s/it]

 20%|██████████████████▏                                                                        | 3/15 [01:44<06:01, 30.15s/it]

 27%|████████████████████████▎                                                                  | 4/15 [02:03<04:45, 25.98s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████            | 13/15 [05:51<00:48, 24.37s/it]

 93%|████████████████████████████████████████████████████████████████████████████████████      | 14/15 [06:13<00:23, 23.57s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:31<00:00, 21.99s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:31<00:00, 26.12s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/T_1998-04.nc


In [18]:
download_MERCATOR(
    Sfiles, "vosaline", starts, ends, x0, x1, y0, y1,outpath+S_out 
)

  0%|                                                                                                   | 0/15 [00:00<?, ?it/s]

  7%|██████                                                                                    | 1/15 [01:41<23:45, 101.79s/it]

 13%|████████████▏                                                                              | 2/15 [02:01<11:36, 53.62s/it]

 20%|██████████████████▏                                                                        | 3/15 [02:21<07:37, 38.11s/it]

 27%|████████████████████████▎                                                                  | 4/15 [02:48<06:12, 33.83s/it]

...

 87%|██████████████████████████████████████████████████████████████████████████████            | 13/15 [05:45<00:39, 19.84s/it]

 93%|████████████████████████████████████████████████████████████████████████████████████      | 14/15 [06:03<00:19, 19.56s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:24<00:00, 19.85s/it]

100%|██████████████████████████████████████████████████████████████████████████████████████████| 15/15 [06:24<00:00, 25.63s/it]

Saved /work/bk1450/b383184/Amazon/Mercator/data/variables/S_1998-04.nc
